# Case 1: mlm_canopy Picard-loop non-convergence (debug/tl-convergence-isolation)

**Research/debug notebook — lives on branch `debug/tl-convergence-isolation`, not intended to be merged.**

Isolates and replays the non-convergence of `CanopyModel.run()`'s inner Picard loop
(T / H2O / CO2 / Tleaf / Tsurf profiles, `pyAPES/canopy/mlm_canopy.py`) that produces
the `"Maximum iterations reached but error tolerable"` / `"Switched to WMA assumption"`
debug messages seen in `Examples/logs/ran_22_k7.log` for the FI-Ran clear-cut run.
This is **case 1** of three (see plan); it is the most frequent (627 occurrences) and
concentrates at night (20:00-04:00), growing from May to September.

To regenerate the captured snapshots this notebook reads:

```bash
rm -rf Examples/debug_captures/case1
PYTHONPATH=. PYAPES_CAPTURE_DIR=Examples/debug_captures/case1 \
    .venv/bin/python Examples/capture_case1.py
```

Each snapshot is a pickle containing:
- `self_snapshot`: deep copy of the `CanopyModel` instance *entering* the failing timestep
  (i.e. before the Picard loop runs) — includes `interception`, `planttypes`, `forestfloor`
  internal warm-start state (LAI/phenology, wet-canopy storage, leaf temperature guesses).
- `forcing`, `parameters`: the exact dicts `CanopyModel.run()` received for that timestep
  (already bundles raw met forcing + current soil-model state — see `pyAPES_MLM.py:347-392`).
- `trajectory`: per-iteration list of `T`, `H2O`, `CO2`, `Tleaf`, `Tsurf` and their errors.
- `outcome`: `'tolerable'` or `'switched_to_wma'`, matching the two failure branches.

In [ ]:
import os
import sys

# Jupyter runs this notebook's kernel with cwd = this file's own directory (Examples/),
# same convention as Demo_multi_layer_model_Ran.ipynb. Paths below are relative to that.
assert os.path.basename(os.getcwd()) == 'debug', (
    f"expected to run with cwd=debug/ (this notebook's own directory), got {os.getcwd()!r} -- adjust paths below if not")
sys.path.insert(0, os.path.abspath('..'))  # repo root, so `import pyAPES` works

CAPTURE_DIR = '../Examples/debug_captures/case1'
REPLAY_DIR = '../Examples/debug_captures/case1_replay'

# set BEFORE importing pyAPES so a fresh replay below also captures a comparison trajectory
os.environ['PYAPES_CAPTURE_DIR'] = REPLAY_DIR

import glob
import pickle
import copy
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

from pyAPES.parameters.mlm_parameters_FI_Ran import gpara


## Load a captured snapshot

In [ ]:
files = sorted(glob.glob(os.path.join(CAPTURE_DIR, '*.pkl')))
print(f'{len(files)} captures found in {CAPTURE_DIR}')
for f in files[:10]:
    print(' ', os.path.basename(f))


In [ ]:
# pick one capture to work with (edit index to explore others / a 'switched_to_wma' case)
CAPTURE_FILE = files[1]
print('Using:', CAPTURE_FILE)

with open(CAPTURE_FILE, 'rb') as fobj:
    capture = pickle.load(fobj)

self_snapshot = capture['self_snapshot']
forcing = capture['forcing']
parameters = capture['parameters']
trajectory = capture['trajectory']
outcome = capture['outcome']

print('date:', parameters['date'], '| outcome:', outcome, '| iterations recorded:', len(trajectory))


## Validation: does replaying the real `CanopyModel.run()` reproduce the same non-convergence?

This is the critical fidelity check before trusting anything else in this notebook: we call the
**unmodified, real** method (imported from the package, not copy-pasted) on the captured snapshot
and confirm it reproduces the same failure trajectory. `PYAPES_CAPTURE_DIR` above points at a
separate `case1_replay/` directory, so this replay will itself dump a fresh capture if it hits the
same non-convergence branch — we load that back and compare it to the original, iteration by iteration.

In [ ]:
replay_snapshot = copy.deepcopy(self_snapshot)  # keep the original capture untouched

out_canopy, out_planttype, out_ffloor, out_groundtype = replay_snapshot.run(
    dt=gpara['dt'],
    forcing=forcing,
    parameters=parameters,
)

replay_files = sorted(glob.glob(os.path.join(REPLAY_DIR, '*.pkl')))
assert replay_files, 'replay did not reproduce a non-convergence — capture may not be self-contained'

with open(replay_files[-1], 'rb') as fobj:
    replay_capture = pickle.load(fobj)

replay_trajectory = replay_capture['trajectory']

print('original outcome:', outcome, '| replay outcome:', replay_capture['outcome'])
print('original iterations:', len(trajectory), '| replay iterations:', len(replay_trajectory))

for key in ['err_t', 'err_h2o', 'err_co2', 'err_Tl', 'err_Ts']:
    orig = np.array([step[key] for step in trajectory])
    repl = np.array([step[key] for step in replay_trajectory])
    match = np.allclose(orig, repl[:len(orig)])
    print(f'  {key}: matches original = {match}')


## Trajectory plots

How the iteration errors and `Tsurf` / mean canopy `T` evolve across the 25 Picard iterations — oscillation vs. slow decay is the key visual diagnostic.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for key in ['err_t', 'err_h2o', 'err_co2', 'err_Tl', 'err_Ts']:
    vals = [step[key] for step in trajectory]
    ax.plot(range(1, len(vals) + 1), vals, marker='o', ms=3, label=key)
ax.axhline(0.01, color='k', ls='--', lw=0.8, label='max_err')
ax.set_xlabel('iteration')
ax.set_ylabel('error')
ax.set_yscale('log')
ax.set_title(f"{parameters['date']}  ({outcome})")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
Tsurf_traj = [step['Tsurf'] for step in trajectory]
Tmean_traj = [np.mean(step['T']) for step in trajectory]
ax.plot(range(1, len(trajectory) + 1), Tsurf_traj, marker='o', ms=3, label='Tsurf (forest floor)')
ax.plot(range(1, len(trajectory) + 1), Tmean_traj, marker='o', ms=3, label='mean(T) canopy air')
ax.set_xlabel('iteration')
ax.set_ylabel('temperature [degC]')
ax.set_title(f"{parameters['date']}  ({outcome})")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## EXPERIMENTAL sandbox — do not treat as source of truth

The cell below is a **copy** of the Picard-loop control logic from `CanopyModel.run()`
(`pyAPES/canopy/mlm_canopy.py`, roughly lines 301-598), with the numerical knobs
(`gam0`, `max_iter`, oscillation-detection window, relaxation floor) exposed as
arguments so candidate fixes can be tried quickly against the same captured snapshot,
without editing source or re-running the full model. It still calls the **real**
submodel objects (`interception`, `planttypes`, `forestfloor`, `radiation`, `micromet`)
carried in the snapshot, so the physics is faithful — only the outer convergence control
is being experimented with here.

**This copy will drift from the source over time.** Any fix validated here must be
ported back into `mlm_canopy.py` as a real patch, then re-verified with a fresh
`capture_case1.py` run showing the debug message disappears/reduces.

In [ ]:
from pyAPES.utils.constants import MOLAR_MASS_H2O, EPS, STEFAN_BOLTZMANN, DEG_TO_KELVIN


def picard_loop(snap, dt, forcing, parameters, max_iter=25, max_err=0.01,
                 gam0=0.5, osc_check_after=5, gam_floor=0.25, verbose=True):
    '''Sandbox copy of CanopyModel.run()'s inner Picard loop. See markdown above.'''
    snap = copy.deepcopy(snap)

    if snap.Switch_Eflow is False:
        snap.micromet.normalized_flow_stats(
            z=snap.z, lad=snap.lad, hc=snap.hc,
            Utop=forcing['wind_speed'] / (forcing['friction_velocity'] + EPS))
    U, ustar = snap.micromet.update_state(forcing['friction_velocity'])

    radiation_profiles = {}
    radiation_params = {'ff_albedo': snap.forestfloor.albedo, 'LAIz': snap.lad * snap.dz}
    radiation_params.update({'radiation_type': 'par'})
    radiation_profiles['par'] = snap.radiation.shortwave_profiles(forcing=forcing, parameters=radiation_params)
    sunlit_fraction = radiation_profiles['par']['sunlit']['fraction']
    if snap.Switch_Ebal:
        radiation_params['radiation_type'] = 'nir'
        radiation_profiles['nir'] = snap.radiation.shortwave_profiles(forcing=forcing, parameters=radiation_params)
        radiation_profiles['sw_absorbed'] = (
            radiation_profiles['par']['sunlit']['absorbed'] * sunlit_fraction
            + radiation_profiles['nir']['sunlit']['absorbed'] * sunlit_fraction
            + radiation_profiles['par']['shaded']['absorbed'] * (1. - sunlit_fraction)
            + radiation_profiles['nir']['shaded']['absorbed'] * (1. - sunlit_fraction))

    gam = gam0
    err_t, err_h2o, err_co2, err_Tl, err_Ts = 999., 999., 999., 999., 999.
    Switch_WMA = snap.Switch_WMA
    T, H2O, CO2, Tleaf, Tsurf = snap._restore(forcing)
    sources = {'h2o': None, 'co2': None, 'sensible_heat': None, 'latent_heat': None, 'fr': None}

    traj = []
    iter_no = 0
    while (err_t > max_err or err_h2o > max_err or err_co2 > max_err
           or err_Tl > max_err or err_Ts > max_err) and iter_no <= max_iter:
        iter_no += 1
        Tleaf_prev = Tleaf.copy()
        Tsurf_prev = Tsurf

        if snap.Switch_Ebal:
            lw_surf = snap.forestfloor.emissivity * STEFAN_BOLTZMANN * np.power(Tsurf + DEG_TO_KELVIN, 4)
            lw_forcing = {'lw_in': forcing['lw_in'], 'lw_up': lw_surf, 'leaf_temperature': Tleaf_prev}
            lw_params = {'LAIz': snap.lad * snap.dz, 'ff_emissivity': snap.forestfloor.emissivity}
            radiation_profiles['lw'] = snap.radiation.longwave_profiles(forcing=lw_forcing, parameters=lw_params)

        for key in sources.keys():
            sources[key] = 0.0 * snap.ones

        interception_forcing = {
            'h2o': H2O, 'wind_speed': U, 'air_temperature': T,
            'air_pressure': forcing['air_pressure'], 'leaf_temperature': Tleaf_prev,
            'precipitation': forcing['precipitation'],
        }
        if snap.Switch_Ebal:
            interception_forcing.update({
                'sw_absorbed': radiation_profiles['sw_absorbed'],
                'lw_radiative_conductance': radiation_profiles['lw']['radiative_conductance'],
                'net_lw_leaf': radiation_profiles['lw']['net_leaf'],
            })
        interception_params = {'LAIz': snap.lad * snap.dz, 'leaf_length': snap.leaf_length}
        interception_controls = {'energy_balance': snap.Switch_Ebal,
                                  'logger_info': f"[sandbox] date: {parameters['date']} iteration: {iter_no}"}

        wetleaf_fluxes = snap.interception.run(dt=dt, forcing=interception_forcing,
                                                parameters=interception_params, controls=interception_controls)
        df = snap.interception.df
        for key in wetleaf_fluxes['sources'].keys():
            sources[key] += wetleaf_fluxes['sources'][key] / snap.dz
        Tleaf = snap.interception.Tl_wet * (1 - df) * snap.lad

        forcing_pt = {
            'h2o': H2O, 'co2': CO2, 'air_temperature': T, 'air_pressure': forcing['air_pressure'],
            'wind_speed': U, 'par': radiation_profiles['par'],
            'average_leaf_temperature': Tleaf_prev, 'wet_leaf_temperature': snap.interception.Tl_wet,
        }
        if snap.Switch_Ebal:
            forcing_pt.update({'nir': radiation_profiles['nir'], 'lw': radiation_profiles['lw']})
        parameters_pt = {'dry_leaf_fraction': snap.interception.df, 'sunlit_fraction': sunlit_fraction}
        controls_pt = {'energy_balance': snap.Switch_Ebal,
                        'logger_info': f"[sandbox] date: {parameters['date']} iteration: {iter_no}"}

        for pt in snap.planttypes:
            pt_stats_i, layer_stats_i = pt.run(forcing=forcing_pt, parameters=parameters_pt, controls=controls_pt)
            sources['co2'] -= layer_stats_i['net_co2']
            sources['h2o'] += layer_stats_i['transpiration']
            sources['sensible_heat'] += layer_stats_i['sensible_heat']
            sources['latent_heat'] += layer_stats_i['latent_heat']
            sources['fr'] += layer_stats_i['fr']
            Tleaf += layer_stats_i['leaf_temperature'] * df * pt.lad
            if pt.LAImax > 0.0:
                rg_tot, rg_layer = pt.growth_respiration(Ta=forcing['air_temperature'])
                sources['co2'] += rg_layer

        Tleaf = Tleaf / (snap.lad + EPS)
        err_Tl = max(abs(Tleaf - Tleaf_prev))

        ff_controls = {'energy_balance': snap.Switch_Ebal,
                        'logger_info': f"[sandbox] date: {parameters['date']} iteration: {iter_no}"}
        ff_params = {
            'reference_height': snap.z[1], 'soil_depth': parameters['soil_depth'],
            'soil_hydraulic_conductivity': parameters['soil_hydraulic_conductivity'][0],
            'soil_thermal_conductivity': parameters['soil_thermal_conductivity'], 'iteration': iter_no,
        }
        ff_forcing = {
            'precipitation_rain': wetleaf_fluxes['throughfall_rain'],
            'precipitation_snow': wetleaf_fluxes['throughfall_snow'],
            'par': radiation_profiles['par']['ground'], 'air_temperature': T[1], 'h2o': H2O[1], 'co2': CO2[1],
            'air_pressure': forcing['air_pressure'], 'wind_speed': U[1], 'friction_velocity': ustar[1],
            'soil_temperature': forcing['soil_temperature'], 'soil_water_potential': forcing['soil_water_potential'][0],
            'soil_volumetric_water': forcing['soil_volumetric_water'], 'soil_volumetric_air': forcing['soil_volumetric_air'],
            'soil_volumetric_ice': forcing['soil_volumetric_ice'], 'soil_pond_storage': forcing['soil_pond_storage'],
        }
        if snap.Switch_Ebal:
            ff_forcing.update({'nir': radiation_profiles['nir']['ground'], 'lw_dn': radiation_profiles['lw']['down'][0]})

        ff_fluxes, ff_states, gt_results = snap.forestfloor.run(dt=dt, forcing=ff_forcing, parameters=ff_params, controls=ff_controls)
        Tsurf = ff_states['surface_temperature']
        err_Ts = abs(Tsurf_prev - Tsurf)

        if Switch_WMA is False:
            if iter_no > 1:
                T_prev2 = T_prev.copy()
            T_prev = T.copy()
            H2O, CO2, T, err_h2o, err_co2, err_t = snap.micromet.scalar_profiles(
                gam, H2O, CO2, T, forcing['air_pressure'], source=sources,
                lbc={'H2O': ff_fluxes['evaporation'] / MOLAR_MASS_H2O, 'CO2': ff_fluxes['net_co2'], 'T': ff_fluxes['sensible_heat']},
                Ebal=snap.Switch_Ebal)

            if iter_no > osc_check_after and np.mean((T_prev - T) ** 2) > np.mean((T_prev2 - T) ** 2):
                T = (T_prev + T) / 2
                gam = max(gam / 2, gam_floor)

            traj.append({'iter_no': iter_no, 'gam': gam, 'Tsurf': Tsurf, 'T_mean': np.mean(T),
                         'err_t': err_t, 'err_h2o': err_h2o, 'err_co2': err_co2, 'err_Tl': err_Tl, 'err_Ts': err_Ts})

            if iter_no == max_iter or any(np.isnan(T)) or any(np.isnan(H2O)) or any(np.isnan(CO2)):
                if max(err_t, err_h2o, err_co2, err_Tl, err_Ts) < 0.05:
                    break
                Switch_WMA = True
                iter_no = 0
                err_t, err_h2o, err_co2, err_Tl, err_Ts = 999., 999., 999., 999., 999.
                T, H2O, CO2, Tleaf, Tsurf = snap._restore(forcing)
        else:
            err_h2o, err_co2, err_t = 0.0, 0.0, 0.0

    if verbose:
        final = traj[-1] if traj else {}
        print(f"gam0={gam0}, osc_check_after={osc_check_after}, gam_floor={gam_floor} "
              f"-> {len(traj)} iterations, final max_err={max(final.get('err_t',0), final.get('err_h2o',0), final.get('err_co2',0), final.get('err_Tl',0), final.get('err_Ts',0)):.4f}")
    return traj


In [ ]:
# baseline (matches source defaults) vs a candidate fix, e.g. stronger initial damping
baseline_traj = picard_loop(self_snapshot, gpara['dt'], forcing, parameters)
gam_test = 0.1
candidate_traj = picard_loop(self_snapshot, gpara['dt'], forcing, parameters, gam0=gam_test, gam_floor=0.15)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, traj_ in [('baseline (gam0=0.5)', baseline_traj), (f'candidate (gam0={gam_test})', candidate_traj)]:
    errs = [max(s['err_t'], s['err_h2o'], s['err_co2'], s['err_Tl'], s['err_Ts']) for s in traj_]
    ax.plot(range(1, len(errs) + 1), errs, marker='o', ms=3, label=label)
ax.axhline(0.01, color='k', ls='--', lw=0.8, label='max_err')
ax.set_yscale('log')
ax.set_xlabel('iteration')
ax.set_ylabel('max error across T/H2O/CO2/Tl/Ts')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## Next steps

If a candidate change in the sandbox above reliably reduces iteration count / final error
across several captured snapshots (not just this one date), port the equivalent minimal
change into `pyAPES/canopy/mlm_canopy.py` directly (not this notebook), then re-run:

```bash
rm -rf Examples/debug_captures/case1
PYTHONPATH=. PYAPES_CAPTURE_DIR=Examples/debug_captures/case1 .venv/bin/python Examples/capture_case1.py
```

and confirm the number of captured non-convergence snapshots for the same window drops.

## Sweep ALL captured problematic snapshots across gamma / gamma_floor

The sandbox above lets you try a candidate `gam0` / `gam_floor` against a single
snapshot. This section runs **every** captured non-convergence snapshot in
`CAPTURE_DIR` through the same `picard_loop` sandbox, so a candidate value can be
checked against all problematic cases at once, not just one date.

Edit `GAM0` and `GAM_FLOOR` below and re-run this section to try different values.

In [ ]:
# --- user-adjustable knobs ---
GAM0 = 0.5
GAM_FLOOR = 0.01
MAX_ITER = 50
MAX_ERR = 0.01
OSC_CHECK_AFTER = 5

import pandas as pd

all_files = sorted(glob.glob(os.path.join(CAPTURE_DIR, '*.pkl')))
print(f'Sweeping {len(all_files)} captured snapshots with gam0={GAM0}, gam_floor={GAM_FLOOR}')

rows = []
for fpath in all_files:
    with open(fpath, 'rb') as fobj:
        cap = pickle.load(fobj)

    snap_i = cap['self_snapshot']
    forcing_i = cap['forcing']
    parameters_i = cap['parameters']
    orig_outcome = cap['outcome']
    orig_iters = len(cap['trajectory'])

    traj_i = picard_loop(
        snap_i, gpara['dt'], forcing_i, parameters_i,
        max_iter=MAX_ITER, max_err=MAX_ERR,
        gam0=GAM0, osc_check_after=OSC_CHECK_AFTER, gam_floor=GAM_FLOOR,
        verbose=False,
    )

    final = traj_i[-1] if traj_i else {}
    final_err = (max(final['err_t'], final['err_h2o'], final['err_co2'], final['err_Tl'], final['err_Ts'])
                 if final else np.nan)
    converged = bool(final) and final_err <= MAX_ERR and len(traj_i) < MAX_ITER

    rows.append({
        'file': os.path.basename(fpath),
        'date': parameters_i['date'],
        'orig_outcome': orig_outcome,
        'orig_iterations': orig_iters,
        'candidate_iterations': len(traj_i),
        'candidate_final_max_err': final_err,
        'candidate_converged': converged,
    })

sweep_df = pd.DataFrame(rows)
sweep_df

In [ ]:
n_converged = int(sweep_df['candidate_converged'].sum())
print(f"{n_converged}/{len(sweep_df)} snapshots converge (max_err <= {MAX_ERR} without hitting max_iter) "
      f"with gam0={GAM0}, gam_floor={GAM_FLOOR}")
sweep_df.sort_values('candidate_final_max_err', ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['tab:green' if c else 'tab:red' for c in sweep_df['candidate_converged']]
ax.bar(range(len(sweep_df)), sweep_df['candidate_final_max_err'], color=colors)
ax.axhline(MAX_ERR, color='k', ls='--', lw=0.8, label='max_err')
ax.set_xticks(range(len(sweep_df)))
ax.set_xticklabels(sweep_df['date'].astype(str), rotation=90, fontsize=7)
ax.set_ylabel('final max error (log scale)')
ax.set_yscale('log')
ax.set_title(f'gam0={GAM0}, gam_floor={GAM_FLOOR}: {n_converged}/{len(sweep_df)} converged')
ax.legend(['max_err'])
plt.tight_layout()
plt.show()